# MTGFlow e SDE-Net t+1/t+6 — heatmap spazio-temporali 2019

Analisi esclusivamente **post-hoc** degli score MTGFlow già prodotti. Il notebook non addestra né modifica MTGFlow. I 16 cluster sono definiti soltanto da latitudine/longitudine; gli score del 2019 determinano esclusivamente il colore dei pixel.

Output principali:
- overview annuale `cluster × giorno`;
- heatmap `località × giorno` per ciascun cluster;
- mappe geografiche del 23–26 aprile e 28–29 giugno 2019;
- heatmap dell'errore SDE-Net separate per forecast t+1 e t+6;
- CSV con cluster, aggregati giornalieri e intensità normalizzata rispetto alla threshold.

In [ ]:
from pathlib import Path
import importlib
import json
import os
import sys

import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
import numpy as np
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'physiq_pv').is_dir():
    raise FileNotFoundError('Avviare il notebook dalla root del repository o da notebooks/.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import physiq_pv.reporting.mtgflow_spatiotemporal as spatial
spatial = importlib.reload(spatial)

MTGFLOW_SEED_DIR = Path(os.environ.get(
    'MTGFLOW_SEED_DIR', ROOT / 'outputs' / 'pvgis_mtgflow' / 'downstream_dense' / 'seed_15'
)).resolve()
MTGFLOW_TEST_CSV = MTGFLOW_SEED_DIR / 'anomaly_scores.csv'
MTGFLOW_TRAIN_CSV = MTGFLOW_SEED_DIR / 'train_anomaly_scores.csv'
TRAINING_STATS_CSV = Path(os.environ.get(
    'MTGFLOW_TRAINING_STATS_CSV',
    ROOT / 'outputs' / 'mtgflow_threshold_sensitivity' / 't_plus_1_and_6' / 'training_iqr_by_location.csv',
)).resolve()
SDE_RUN_NAME = 'pvgis_stgnn_paper_faithful_gaussian_detector_mtgflow_ep60_h1-2-3-4-5-6_direct_seed1'
FORECAST_HORIZONS = (1, 6)
SDE_PREDICTIONS_CSV = Path(os.environ.get(
    'SDE_PREDICTIONS_CSV', ROOT / 'outputs' / SDE_RUN_NAME / 'predictions.csv'
)).resolve()
PVGIS_2019_PATH = Path(os.environ.get(
    'PVGIS_2019_PATH',
    '/data/SentinelPV/pvgis_data/data/pvgis_summed_irradiance/piedmont_pvgis_2019.nc',
)).resolve()
OUT_DIR = Path(os.environ.get(
    'MTGFLOW_SPATIOTEMPORAL_OUT_DIR',
    ROOT / 'outputs' / 'mtgflow_spatiotemporal_2019' / 't_plus_1_and_6',
)).resolve()
FIGURE_DIR = OUT_DIR / 'figures'
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

N_GEO_CLUSTERS = 16
GEO_K_NEIGHBORS = 8
EXPECTED_LOCATIONS = int(os.environ.get('EXPECTED_LOCATIONS', '1149'))
DAYTIME_THRESHOLD_WM2 = 10.0
CSV_CHUNKSIZE = 500_000
APRIL_EVENT_DATES = ('2019-04-23', '2019-04-24', '2019-04-25', '2019-04-26')
JUNE_EVENT_DATES = ('2019-06-28', '2019-06-29')
FULL_DATES = pd.date_range('2019-01-01', '2019-12-31', freq='D')

print('MTGFlow test:', MTGFLOW_TEST_CSV)
print('SDE t+1 e t+6:', SDE_PREDICTIONS_CSV)
print('PVGIS 2019 :', PVGIS_2019_PATH)
print('Output     :', OUT_DIR)

## 1. Dati e definizione dell'anomalia

La decisione binaria usa la threshold MTGFlow salvata per ogni località, equivalente al baseline del paper con `k=1.5`:

`is_anomaly = anomaly_score >= threshold`

L'intensità visualizzata è il superamento in unità di IQR storico:

`positive_excess_iqr = max((anomaly_score - threshold) / IQR_train, 0)`

Q1, Q3 e IQR provengono dal training storico, mai dagli score 2019. Sono usate soltanto le ore con POA ricostruita maggiore di 10 W/m².

In [ ]:
for required in (MTGFLOW_TEST_CSV, SDE_PREDICTIONS_CSV, PVGIS_2019_PATH):
    if not required.is_file():
        raise FileNotFoundError(required)

locations, pvgis_times, poa_by_location_time = spatial.load_pvgis_spatial_context(
    PVGIS_2019_PATH
)
if EXPECTED_LOCATIONS is not None and len(locations) != EXPECTED_LOCATIONS:
    raise ValueError(f'Attese {EXPECTED_LOCATIONS} località PVGIS, trovate {len(locations)}.')
if set(pvgis_times.year) != {2019}:
    raise ValueError('Il NetCDF deve contenere esclusivamente il 2019.')

saved_thresholds = spatial.read_saved_thresholds(
    MTGFLOW_TEST_CSV, chunksize=CSV_CHUNKSIZE
)
per_site_training_paths = sorted(MTGFLOW_SEED_DIR.glob('*/train_scores.csv'))
threshold_table = spatial.build_threshold_table(
    saved_thresholds,
    cached_statistics=TRAINING_STATS_CSV,
    per_site_training_paths=per_site_training_paths,
    aggregate_training_path=MTGFLOW_TRAIN_CSV,
)
if set(locations['location']) != set(threshold_table['location']):
    missing_scores = sorted(set(locations['location']) - set(threshold_table['location']))[:10]
    missing_coordinates = sorted(set(threshold_table['location']) - set(locations['location']))[:10]
    raise ValueError(
        f'Disallineamento località: senza score={missing_scores}, senza coordinate={missing_coordinates}'
    )
threshold_table.to_csv(OUT_DIR / 'threshold_iqr_by_location.csv', index=False)
print('Località:', len(locations))
print('Timestamp PVGIS:', len(pvgis_times))
print('Sorgente IQR:', threshold_table['threshold_statistics_source'].unique().tolist())
display(threshold_table.head())

## 2. Cluster esclusivamente geografici

Il clustering Ward è vincolato al grafo delle 8 località più vicine. Gli anomaly score non entrano nel clustering.

In [ ]:
clusters = spatial.build_geographic_clusters(
    locations, n_clusters=N_GEO_CLUSTERS, n_neighbors=GEO_K_NEIGHBORS
)
clusters.to_csv(OUT_DIR / 'geographic_clusters.csv', index=False)
cluster_sizes = clusters.groupby('geo_cluster', observed=True).size().rename('n_locations')
display(cluster_sizes.to_frame().T)

cluster_map_path = FIGURE_DIR / 'geographic_clusters.png'
fig, axis = plt.subplots(figsize=(8, 8))
scatter = axis.scatter(
    clusters['longitude'], clusters['latitude'], c=clusters['geo_cluster'],
    cmap='tab20', marker='s', s=18, linewidths=0,
)
for cluster_id, center in clusters.groupby('geo_cluster')[['longitude', 'latitude']].mean().iterrows():
    axis.text(center['longitude'], center['latitude'], str(cluster_id), ha='center', va='center', fontsize=8)
axis.set(
    title='PVGIS — 16 cluster geografici KNN-Ward',
    xlabel='Longitudine', ylabel='Latitudine',
)
axis.set_aspect(1.0 / np.cos(np.deg2rad(clusters['latitude'].mean())))
axis.grid(alpha=0.15)
fig.savefig(cluster_map_path, dpi=180, bbox_inches='tight')
plt.show()
print('Mappa cluster:', cluster_map_path)

## 3. Aggregazione dell'intero 2019

Il CSV viene letto a chunk. Ogni pixel giornaliero conserva sia la frazione di ore diurne anomale sia il massimo superamento della threshold in unità IQR.

In [ ]:
daily = spatial.aggregate_daily_scores(
    MTGFLOW_TEST_CSV, threshold_table, locations, pvgis_times,
    poa_by_location_time, daytime_threshold_wm2=DAYTIME_THRESHOLD_WM2,
    chunksize=CSV_CHUNKSIZE,
)
aggregation_metadata = dict(daily.attrs)
daily_with_geo = daily.merge(
    clusters, on='location', how='left', validate='many_to_one'
)
cluster_daily = spatial.aggregate_clusters(daily, clusters)
daily_with_geo.to_csv(OUT_DIR / 'daily_location_anomalies_2019.csv', index=False)
cluster_daily.to_csv(OUT_DIR / 'daily_cluster_anomalies_2019.csv', index=False)
print(f"Righe MTGFlow lette: {aggregation_metadata['source_rows']:,}")
print(f"Righe diurne usate: {aggregation_metadata['retained_rows']:,}")
print('Celle località-giorno:', len(daily_with_geo))
display(cluster_daily.head())

forecast_daily_by_horizon = {}
forecast_daily_with_geo_by_horizon = {}
forecast_cluster_daily_by_horizon = {}
forecast_metadata_by_horizon = {}
for horizon_hours in FORECAST_HORIZONS:
    forecast_daily = spatial.aggregate_daily_forecast_errors(
        SDE_PREDICTIONS_CSV, horizon_hours=horizon_hours,
        daytime_threshold_wm2=DAYTIME_THRESHOLD_WM2, chunksize=CSV_CHUNKSIZE,
    )
    forecast_metadata = dict(forecast_daily.attrs)
    forecast_daily_with_geo = forecast_daily.merge(
        clusters, on='location', how='left', validate='many_to_one'
    )
    if forecast_daily_with_geo['geo_cluster'].isna().any():
        raise ValueError('Almeno una località SDE non ha coordinate o cluster geografico.')
    forecast_cluster_daily = spatial.aggregate_forecast_error_clusters(
        forecast_daily, clusters
    )
    forecast_daily_by_horizon[horizon_hours] = forecast_daily
    forecast_daily_with_geo_by_horizon[horizon_hours] = forecast_daily_with_geo
    forecast_cluster_daily_by_horizon[horizon_hours] = forecast_cluster_daily
    forecast_metadata_by_horizon[horizon_hours] = forecast_metadata
    forecast_daily_with_geo.to_csv(
        OUT_DIR / f'daily_location_forecast_errors_t_plus_{horizon_hours}.csv', index=False
    )
    forecast_cluster_daily.to_csv(
        OUT_DIR / f'daily_cluster_forecast_errors_t_plus_{horizon_hours}.csv', index=False
    )
    print(
        f"Righe SDE t+{horizon_hours}: {forecast_metadata['horizon_rows']:,}; "
        f"diurne usate: {forecast_metadata['retained_rows']:,}; "
        f"altri orizzonti esclusi: {forecast_metadata['skipped_other_horizons']:,}"
    )
    display(forecast_cluster_daily.head())

In [ ]:
month_starts = pd.date_range('2019-01-01', '2019-12-01', freq='MS')
month_positions = [FULL_DATES.get_loc(date) for date in month_starts]
month_labels = [date.strftime('%b') for date in month_starts]

def _finite_vmax(values, quantile=0.99, floor=1.0):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values) & (values > 0)]
    return max(float(np.quantile(values, quantile)), floor) if values.size else floor

def _format_heatmap_axis(axis, matrix, title, ylabel):
    axis.set_title(title)
    axis.set_ylabel(ylabel)
    axis.set_xticks(month_positions, month_labels)
    n_rows = len(matrix.index)
    step = max(1, n_rows // 12)
    positions = np.arange(0, n_rows, step)
    axis.set_yticks(positions, [str(matrix.index[i]) for i in positions])

def _matrix(frame, index, value):
    return frame.pivot(index=index, columns='date', values=value).reindex(columns=FULL_DATES)

## 4. Heatmap annuale: cluster × giorno

In [ ]:
overview_fraction = _matrix(cluster_daily, 'geo_cluster', 'anomaly_fraction')
overview_severity = _matrix(cluster_daily, 'geo_cluster', 'max_positive_excess_iqr')
overview_path = FIGURE_DIR / 'annual_cluster_heatmap_2019.png'
fig, axes = plt.subplots(2, 1, figsize=(18, 9), sharex=True)
fraction_image = axes[0].imshow(
    overview_fraction.to_numpy(), aspect='auto', interpolation='nearest',
    cmap='Reds', vmin=0.0, vmax=1.0,
)
_format_heatmap_axis(axes[0], overview_fraction, 'Frazione di celle località-ora anomale', 'Cluster geografico')
fig.colorbar(fraction_image, ax=axes[0], label='Frazione anomala')
severity_vmax = _finite_vmax(overview_severity.to_numpy())
severity_image = axes[1].imshow(
    overview_severity.to_numpy(), aspect='auto', interpolation='nearest',
    cmap='magma', vmin=0.0, vmax=severity_vmax,
)
_format_heatmap_axis(axes[1], overview_severity, 'Massimo superamento della threshold', 'Cluster geografico')
axes[1].set_xlabel('Giorno del 2019')
fig.colorbar(severity_image, ax=axes[1], label='Superamento [IQR storico]')
fig.suptitle('MTGFlow — distribuzione spazio-temporale annuale', y=1.01)
fig.tight_layout()
fig.savefig(overview_path, dpi=180, bbox_inches='tight')
plt.show()
print('Overview:', overview_path)

## 5. Heatmap annuali dell'errore SDE-Net direct t+1 e t+6

MAE, RMSE e bias sono aggregati separatamente dalle righe diurne con `horizon_hours == 1` e `horizon_hours == 6`. Le heatmap MTGFlow precedenti restano indipendenti dall'orizzonte.

In [ ]:
forecast_overview_paths = {}
for horizon_hours in FORECAST_HORIZONS:
    forecast_cluster_daily = forecast_cluster_daily_by_horizon[horizon_hours]
    forecast_overview = {
        metric: _matrix(forecast_cluster_daily, 'geo_cluster', metric)
        for metric in ('mae', 'rmse', 'bias')
    }
    mae_vmax = _finite_vmax(forecast_overview['mae'].to_numpy())
    rmse_vmax = _finite_vmax(forecast_overview['rmse'].to_numpy())
    bias_vmax = _finite_vmax(np.abs(forecast_overview['bias'].to_numpy()))
    fig, axes = plt.subplots(3, 1, figsize=(18, 12), sharex=True)
    for axis, metric, title, cmap, vmin, vmax, label in (
        (axes[0], 'mae', f'MAE giornaliero t+{horizon_hours}', 'viridis', 0.0, mae_vmax, 'MAE [W]'),
        (axes[1], 'rmse', f'RMSE giornaliero t+{horizon_hours}', 'magma', 0.0, rmse_vmax, 'RMSE [W]'),
        (axes[2], 'bias', f'Bias giornaliero t+{horizon_hours}', 'coolwarm', -bias_vmax, bias_vmax, 'Bias [W]'),
    ):
        image = axis.imshow(
            forecast_overview[metric].to_numpy(), aspect='auto', interpolation='nearest',
            cmap=cmap, vmin=vmin, vmax=vmax,
        )
        _format_heatmap_axis(axis, forecast_overview[metric], title, 'Cluster geografico')
        fig.colorbar(image, ax=axis, label=label)
    axes[-1].set_xlabel('Giorno del 2019')
    fig.suptitle(
        f'SDE-Net direct — errore spazio-temporale annuale — forecast t+{horizon_hours}',
        y=1.01,
    )
    fig.tight_layout()
    forecast_overview_path = (
        FIGURE_DIR / f'annual_cluster_forecast_error_heatmap_t_plus_{horizon_hours}.png'
    )
    fig.savefig(forecast_overview_path, dpi=180, bbox_inches='tight')
    forecast_overview_paths[horizon_hours] = forecast_overview_path
    plt.show()
    print(f'Heatmap errore t+{horizon_hours}:', forecast_overview_path)

## 6. Heatmap annuali: località × giorno per cluster

In [ ]:
cluster_heatmap_paths = {}
global_severity_vmax = _finite_vmax(daily_with_geo['max_positive_excess_iqr'])
for cluster_id in sorted(clusters['geo_cluster'].unique()):
    cluster_locations = (
        clusters.loc[clusters['geo_cluster'] == cluster_id]
        .sort_values('within_cluster_order')['location'].tolist()
    )
    subset = daily_with_geo[daily_with_geo['geo_cluster'] == cluster_id]
    fraction = _matrix(subset, 'location', 'anomaly_fraction').reindex(cluster_locations)
    severity = _matrix(subset, 'location', 'max_positive_excess_iqr').reindex(cluster_locations)
    fig, axes = plt.subplots(2, 1, figsize=(18, 10), sharex=True)
    image_fraction = axes[0].imshow(
        fraction.to_numpy(), aspect='auto', interpolation='nearest',
        cmap='Reds', vmin=0.0, vmax=1.0,
    )
    _format_heatmap_axis(axes[0], fraction, 'Frazione di ore diurne anomale', 'Location ID')
    fig.colorbar(image_fraction, ax=axes[0], label='Frazione anomala')
    image_severity = axes[1].imshow(
        severity.to_numpy(), aspect='auto', interpolation='nearest',
        cmap='magma', vmin=0.0, vmax=global_severity_vmax,
    )
    _format_heatmap_axis(axes[1], severity, 'Massimo superamento della threshold', 'Location ID')
    axes[1].set_xlabel('Giorno del 2019')
    fig.colorbar(image_severity, ax=axes[1], label='Superamento [IQR storico]')
    fig.suptitle(f'MTGFlow — cluster geografico {cluster_id} — intero 2019', y=1.01)
    fig.tight_layout()
    path = FIGURE_DIR / f'annual_location_heatmap_cluster_{cluster_id:02d}.png'
    fig.savefig(path, dpi=170, bbox_inches='tight')
    plt.close(fig)
    cluster_heatmap_paths[int(cluster_id)] = path
print('Heatmap MTGFlow per cluster create:', len(cluster_heatmap_paths))

forecast_cluster_heatmap_paths = {horizon: {} for horizon in FORECAST_HORIZONS}
for horizon_hours in FORECAST_HORIZONS:
    forecast_daily_with_geo = forecast_daily_with_geo_by_horizon[horizon_hours]
    global_mae_vmax = _finite_vmax(forecast_daily_with_geo['mae'])
    global_rmse_vmax = _finite_vmax(forecast_daily_with_geo['rmse'])
    global_bias_vmax = _finite_vmax(np.abs(forecast_daily_with_geo['bias']))
    for cluster_id in sorted(clusters['geo_cluster'].unique()):
        cluster_locations = (
            clusters.loc[clusters['geo_cluster'] == cluster_id]
            .sort_values('within_cluster_order')['location'].tolist()
        )
        subset = forecast_daily_with_geo[
            forecast_daily_with_geo['geo_cluster'] == cluster_id
        ]
        matrices = {
            metric: _matrix(subset, 'location', metric).reindex(cluster_locations)
            for metric in ('mae', 'rmse', 'bias')
        }
        fig, axes = plt.subplots(3, 1, figsize=(18, 14), sharex=True)
        for axis, metric, title, cmap, vmin, vmax, label in (
            (axes[0], 'mae', f'MAE giornaliero t+{horizon_hours}', 'viridis', 0.0, global_mae_vmax, 'MAE [W]'),
            (axes[1], 'rmse', f'RMSE giornaliero t+{horizon_hours}', 'magma', 0.0, global_rmse_vmax, 'RMSE [W]'),
            (axes[2], 'bias', f'Bias giornaliero t+{horizon_hours}', 'coolwarm', -global_bias_vmax, global_bias_vmax, 'Bias [W]'),
        ):
            image = axis.imshow(
                matrices[metric].to_numpy(), aspect='auto', interpolation='nearest',
                cmap=cmap, vmin=vmin, vmax=vmax,
            )
            _format_heatmap_axis(axis, matrices[metric], title, 'Location ID')
            fig.colorbar(image, ax=axis, label=label)
        axes[-1].set_xlabel('Giorno del 2019')
        fig.suptitle(
            f'SDE-Net direct t+{horizon_hours} — cluster geografico {cluster_id}', y=1.01
        )
        fig.tight_layout()
        path = (
            FIGURE_DIR
            / f'annual_location_forecast_error_t_plus_{horizon_hours}_cluster_{cluster_id:02d}.png'
        )
        fig.savefig(path, dpi=170, bbox_inches='tight')
        plt.close(fig)
        forecast_cluster_heatmap_paths[horizon_hours][int(cluster_id)] = path
    print(
        f'Heatmap errore SDE t+{horizon_hours} per cluster create:',
        len(forecast_cluster_heatmap_paths[horizon_hours]),
    )

## 7. Mappe geografiche: eventi di aprile e giugno

Ogni località è rappresentata come un pixel quadrato. Il grigio indica nessun superamento della threshold nelle ore diurne; il rosso indica anomalia e la sua intensità è il massimo superamento giornaliero in unità IQR.

In [ ]:
def build_event_map(event_name, event_dates):
    selected_dates = pd.DatetimeIndex(event_dates)
    event = daily_with_geo[daily_with_geo['date'].isin(selected_dates)].copy()
    expected = pd.MultiIndex.from_product(
        [clusters['location'], selected_dates], names=['location', 'date']
    )
    event = (
        event.set_index(['location', 'date'])
        .reindex(expected).reset_index()
        .drop(columns=['latitude', 'longitude', 'geo_cluster', 'within_cluster_order'], errors='ignore')
        .merge(clusters, on='location', how='left', validate='many_to_one')
    )
    event['max_positive_excess_iqr'] = event['max_positive_excess_iqr'].fillna(0.0)
    event['n_anomalous'] = event['n_anomalous'].fillna(0).astype(int)
    event.to_csv(OUT_DIR / f'{event_name}_daily_location_map.csv', index=False)
    positive = event.loc[event['max_positive_excess_iqr'] > 0, 'max_positive_excess_iqr']
    vmax = _finite_vmax(positive)
    norm = Normalize(vmin=0.0, vmax=vmax)
    n_dates = len(selected_dates)
    n_columns = min(2, n_dates)
    n_rows = int(np.ceil(n_dates / n_columns))
    fig, axes = plt.subplots(n_rows, n_columns, figsize=(7 * n_columns, 7 * n_rows), squeeze=False)
    for axis, date in zip(axes.flat, selected_dates):
        frame = event[event['date'] == date]
        anomalous = frame['max_positive_excess_iqr'] > 0
        axis.scatter(
            frame.loc[~anomalous, 'longitude'], frame.loc[~anomalous, 'latitude'],
            color='#dedede', marker='s', s=22, linewidths=0, label='Normale',
        )
        axis.scatter(
            frame.loc[anomalous, 'longitude'], frame.loc[anomalous, 'latitude'],
            c=frame.loc[anomalous, 'max_positive_excess_iqr'], cmap='Reds', norm=norm,
            marker='s', s=26, edgecolors='black', linewidths=0.15, label='Anomala',
        )
        anomalous_fraction = float((frame['n_anomalous'] > 0).mean())
        axis.set(
            title=f"{date:%Y-%m-%d} — località anomale {anomalous_fraction:.1%}",
            xlabel='Longitudine', ylabel='Latitudine',
        )
        axis.set_aspect(1.0 / np.cos(np.deg2rad(frame['latitude'].mean())))
        axis.grid(alpha=0.12)
    for axis in axes.flat[n_dates:]:
        axis.set_visible(False)
    colorbar = fig.colorbar(ScalarMappable(norm=norm, cmap='Reds'), ax=axes.ravel().tolist(), shrink=0.8)
    colorbar.set_label('Massimo superamento giornaliero [IQR storico]')
    fig.suptitle(f'MTGFlow — distribuzione geografica — {event_name.replace("_", " " )}', y=0.995)
    path = FIGURE_DIR / f'{event_name}_geographic_anomaly_map.png'
    fig.savefig(path, dpi=180, bbox_inches='tight')
    plt.close(fig)
    return path

april_map_path = build_event_map('april_dust_23_26', APRIL_EVENT_DATES)
june_map_path = build_event_map('june_extreme_28_29', JUNE_EVENT_DATES)
display(Image(filename=str(april_map_path)))
display(Image(filename=str(june_map_path)))
print('Aprile:', april_map_path)
print('Giugno:', june_map_path)

def build_forecast_event_map(event_name, event_dates, horizon_hours, forecast_daily_with_geo):
    selected_dates = pd.DatetimeIndex(event_dates)
    event = forecast_daily_with_geo[
        forecast_daily_with_geo['date'].isin(selected_dates)
    ].copy()
    expected = pd.MultiIndex.from_product(
        [clusters['location'], selected_dates], names=['location', 'date']
    )
    event = (
        event.set_index(['location', 'date']).reindex(expected).reset_index()
        .drop(columns=['latitude', 'longitude', 'geo_cluster', 'within_cluster_order'], errors='ignore')
        .merge(clusters, on='location', how='left', validate='many_to_one')
    )
    if event[['mae', 'rmse', 'bias']].isna().any().any():
        raise ValueError(f'Predizioni t+{horizon_hours} mancanti per {event_name}.')
    event.to_csv(
        OUT_DIR / f'{event_name}_daily_location_forecast_error_t_plus_{horizon_hours}.csv',
        index=False,
    )
    vmax = _finite_vmax(event['rmse'])
    norm = Normalize(vmin=0.0, vmax=vmax)
    n_dates = len(selected_dates)
    n_columns = min(2, n_dates)
    n_rows = int(np.ceil(n_dates / n_columns))
    fig, axes = plt.subplots(
        n_rows, n_columns, figsize=(7 * n_columns, 7 * n_rows), squeeze=False
    )
    for axis, date in zip(axes.flat, selected_dates):
        frame = event[event['date'] == date]
        total = frame['n_forecasts'].sum()
        regional_mae = frame['sum_abs_error'].sum() / total
        regional_rmse = np.sqrt(frame['sum_squared_error'].sum() / total)
        axis.scatter(
            frame['longitude'], frame['latitude'], c=frame['rmse'],
            cmap='magma', norm=norm, marker='s', s=26, linewidths=0,
        )
        axis.set(
            title=f'{date:%Y-%m-%d} — MAE {regional_mae:.2f} W — RMSE {regional_rmse:.2f} W',
            xlabel='Longitudine', ylabel='Latitudine',
        )
        axis.set_aspect(1.0 / np.cos(np.deg2rad(frame['latitude'].mean())))
        axis.grid(alpha=0.12)
    for axis in axes.flat[n_dates:]:
        axis.set_visible(False)
    colorbar = fig.colorbar(
        ScalarMappable(norm=norm, cmap='magma'), ax=axes.ravel().tolist(), shrink=0.8
    )
    colorbar.set_label(f'RMSE forecast t+{horizon_hours} [W]')
    fig.suptitle(
        f'SDE-Net direct t+{horizon_hours} — errore geografico — {event_name.replace("_", " " )}',
        y=0.995,
    )
    path = FIGURE_DIR / f'{event_name}_geographic_forecast_error_t_plus_{horizon_hours}.png'
    fig.savefig(path, dpi=180, bbox_inches='tight')
    plt.close(fig)
    return path

forecast_event_maps = {}
for horizon_hours in FORECAST_HORIZONS:
    forecast_daily_with_geo = forecast_daily_with_geo_by_horizon[horizon_hours]
    april_forecast_map = build_forecast_event_map(
        'april_dust_23_26', APRIL_EVENT_DATES, horizon_hours, forecast_daily_with_geo
    )
    june_forecast_map = build_forecast_event_map(
        'june_extreme_28_29', JUNE_EVENT_DATES, horizon_hours, forecast_daily_with_geo
    )
    forecast_event_maps[horizon_hours] = {
        'april': april_forecast_map, 'june': june_forecast_map,
    }
    display(Image(filename=str(april_forecast_map)))
    display(Image(filename=str(june_forecast_map)))
    print(f'Errore t+{horizon_hours} aprile:', april_forecast_map)
    print(f'Errore t+{horizon_hours} giugno:', june_forecast_map)

## 8. Metadati ed elenco degli output

In [ ]:
metadata = {
    'analysis': 'posthoc_mtgflow_spatiotemporal_and_sde_t1_t6_error',
    'training_rerun': False,
    'mtgflow_scores': str(MTGFLOW_TEST_CSV),
    'sde_predictions': str(SDE_PREDICTIONS_CSV),
    'forecast_horizons_hours': list(FORECAST_HORIZONS),
    'pvgis_2019': str(PVGIS_2019_PATH),
    'n_locations': int(len(locations)),
    'n_geo_clusters': int(N_GEO_CLUSTERS),
    'geo_k_neighbors': int(GEO_K_NEIGHBORS),
    'clustering_features': ['latitude', 'longitude'],
    'clustering_uses_anomaly_scores': False,
    'binary_rule': 'anomaly_score >= saved_threshold',
    'continuous_intensity': 'max((anomaly_score - saved_threshold) / training_iqr, 0)',
    'threshold_statistics_source': threshold_table['threshold_statistics_source'].unique().tolist(),
    'daytime_threshold_wm2': DAYTIME_THRESHOLD_WM2,
    'score_rows': aggregation_metadata,
    'forecast_rows_by_horizon': forecast_metadata_by_horizon,
    'april_event_dates': list(APRIL_EVENT_DATES),
    'june_event_dates': list(JUNE_EVENT_DATES),
}
metadata_path = OUT_DIR / 'analysis_metadata.json'
metadata_path.write_text(json.dumps(metadata, indent=2, default=str), encoding='utf-8')

output_files = sorted(path.relative_to(OUT_DIR) for path in OUT_DIR.rglob('*') if path.is_file())
print(f'Creati {len(output_files)} file in {OUT_DIR}:')
for path in output_files:
    print(' -', path)

## Interpretazione

- Una colonna rossa limitata a poche righe suggerisce un'anomalia locale.
- Una colonna rossa estesa a molte località dello stesso cluster suggerisce un evento spazialmente coerente.
- Lo stesso giorno rosso in più cluster indica un evento regionale.
- Le mappe di aprile e giugno mostrano direttamente se il superamento è concentrato o distribuito sul Piemonte.
- Le heatmap MAE/RMSE/bias e le mappe geografiche dell'errore sono generate separatamente per forecast direct t+1 e t+6.
- La mappa MTGFlow va letta insieme alle due mappe dell'errore: la prima localizza le anomalie, le altre mostrano come cresce o cambia l'errore tra t+1 e t+6.

Questa è una visualizzazione spaziale degli output di un detector location-wise: non trasforma MTGFlow in un modello spazio-temporale e non implica causalità tra località.